# Exploratory evaluation

This notebook inspects a saved evaluation result set. Set `LCL_EVALUATION_DIR` to select one explicitly; otherwise the latest completed result set is used.

In [ ]:
from pathlib import Path
import os
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "implementations").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the LCL-evaluation repository.")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "src"))

from evaluation_data import (
    STAGES,
    STAGE_NAMES,
    load_loc,
    load_usage,
    resolve_evaluation_dir,
    stage_usage_paths,
    summarize_benchmark,
    summarize_scalability,
)

EVALUATION_DIR = resolve_evaluation_dir(ROOT, os.environ.get("LCL_EVALUATION_DIR"))
EVALUATION_DIR

## Benchmark summary

In [ ]:
benchmark = summarize_benchmark(EVALUATION_DIR)
benchmark

## Scalability summary

In [ ]:
scalability = summarize_scalability(EVALUATION_DIR)
scalability

## Lines changed

In [ ]:
loc = load_loc(EVALUATION_DIR)
loc

## AutoML output comparison

In [ ]:
SUBSETS = ("spring", "winter")
SPLITS = ("KFold", "Random")


def find_report(directory: Path, filename: str) -> Path:
    matches = list(directory.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one {filename} in {directory}, found {len(matches)}")
    return matches[0]


def summarize_report(implementation: str, subset: str, split: str, report: dict) -> dict:
    return {
        "implementation": implementation,
        "subset": subset,
        "split": split,
        "rmse_mean": report["scores"]["rmse"]["mean"],
        "rmse_std": report["scores"]["rmse"]["std"],
        "r_squared_mean": report["scores"]["r_squared"]["mean"],
        "r_squared_std": report["scores"]["r_squared"]["std"],
        "fit_time_mean": report["fit_time"]["mean"],
        "fit_time_std": report["fit_time"]["std"],
        "params": report["params"],
    }


localize_reports = EVALUATION_DIR / "benchmark" / "localize" / "run-01" / "reports"
jupyter_reports = EVALUATION_DIR / "benchmark" / "jupyter" / "run-01" / "reports"
rows = []
for subset in SUBSETS:
    for split in SPLITS:
        localize_path = find_report(
            localize_reports, f"{subset}-ExampleModel-{split}Split-results.pkl"
        )
        jupyter_path = find_report(
            jupyter_reports, f"Results_ExampleModel-{split}Split-{subset}Subset.pkl"
        )
        localize_report = joblib.load(localize_path)["model_data"]["reports"][0]
        jupyter_report = joblib.load(jupyter_path)[0]
        rows.append(summarize_report("LOCALIZE", subset, split, localize_report))
        rows.append(summarize_report("Jupyter", subset, split, jupyter_report))

model_comparison = pd.DataFrame(rows)
model_comparison

## Raw resource traces

In [ ]:
RUN = 1
STAGE = "gridsearch"
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True, constrained_layout=True)

for implementation, label in (("localize", "LOCALIZE"), ("jupyter", "Jupyter")):
    run_dir = EVALUATION_DIR / "benchmark" / implementation / f"run-{RUN:02d}"
    for index, path in enumerate(stage_usage_paths(run_dir, STAGE)):
        frame = load_usage(path)
        trace_label = label if index == 0 else None
        axes[0].plot(frame["t"], frame["cpu_cores"], alpha=0.7, label=trace_label)
        axes[1].plot(frame["t"], frame["ram_mb"], alpha=0.7, label=trace_label)

axes[0].set_ylabel("CPU [cores]")
axes[1].set_ylabel("Memory [MB]")
axes[1].set_xlabel("Elapsed time [s]")
axes[0].set_title(f"{STAGE_NAMES[STAGE]} resource traces, run {RUN}")
for ax in axes:
    ax.grid(linestyle=":", linewidth=0.5)
    ax.legend()
plt.show()

## Alternative scalability views

In [ ]:
size_order = ["1x", "5x", "10x"]
scalability_totals = (
    scalability.groupby("size")[["core_seconds", "wall_seconds"]]
    .sum()
    .loc[size_order]
)
scalability_relative = scalability_totals.div(scalability_totals.loc["1x"])
display(scalability_totals)
display(scalability_relative)

axes = scalability_totals.plot.barh(
    subplots=True, figsize=(9, 4), sharey=True, grid=True, legend=False
)
axes[0].set_xlabel("CPU time [core·s]")
axes[1].set_xlabel("Wall time [s]")
plt.tight_layout()
plt.show()